# 28 · P0 roadmap: wear на дефолтах + inference-контракт + финальная таблица

Задачи дорожной карты (§20 PLAN_OTHER_TASKS.md), приоритет P0:

1. **Wear (износ инфраструктуры)**: переобучить слой 2 на текущих дефолтах
   `1000 / 0.03 / 10` (вместо старых `500 / 0.05 / 6`) и пересчитать отчёт
   (метрики 3–30 дней); сохранить `models/tte_wear_*`.
2. **Inference-контракт для backend**: единый файл-утилита
   `research/inference_contract.py` — модель + калибровка risk30 + топ-K
   (контракт §19), демонстрация на holdout-предсказаниях.
3. **Финальная таблица метрик** (`inference_contract.final_metrics`) по задачам
   (fire/access/sensor/wear) на holdout ≥ 2025-07-01.
4. **Drift-мониторинг (P1)**: процентили p24/risk30 по кварталам.

Протокол валидации неизменен: обучение ≤ 2024-12-31, val 2025H1, holdout ≥ 2025-07-01;
кампанийные недели исключены.

In [1]:
%matplotlib inline
import pathlib
import sys
import time

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

import data_utils as du
import tte_pipeline as tte
from tte_experiments import (baseline_survival_by_type, fit_discrete_hazard,
                             pr_at_horizon, predict_survival_discrete,
                             survival_metrics, verify_no_lookup)
import inference_contract as ic

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

DATASET = du.find_dataset_dir()
MODELS = RESEARCH / "models"
print("dataset:", DATASET)
print("границы:", tte.TRAIN_END.date(), "->", tte.HOLDOUT_START.date())

# Параметры, принятые §15/§20 (те же, что для финальной fire-модели task_tte.py)
CFG = dict(
    n_starts=15_000, n_fault=10_000, n_nonfault=10_000,
    n_val=6_000, n_hold=20_000,
    iters=1000, lr=0.03, depth=10, seed=42,
    task_type="GPU",   # GPU (RTX 3080); финальная fire на CPU — см. §15/§28
    es=300,
)
print("CFG:", CFG)


dataset: D:\Downloads_D\lct_a8\research\dataset
границы: 2024-12-31 -> 2025-07-01
CFG: {'n_starts': 15000, 'n_fault': 10000, 'n_nonfault': 10000, 'n_val': 6000, 'n_hold': 20000, 'iters': 1000, 'lr': 0.03, 'depth': 10, 'seed': 42, 'task_type': 'GPU', 'es': 300}


In [2]:
# 1. Wear: таблица субъектов (канал x 6ч-бакет), событие = старт серии неисправностей
subjects = pd.read_parquet(DATASET / "tte_subjects.parquet")
print("субъектов:", f"{subjects.shape[0]:,}", "| колонок:", subjects.shape[1])
print("событие в 30д: %.1f%% | медиана obs: %.1f дн | цензур: %.1f%%" % (
    subjects["event_flag"].mean() * 100, subjects["obs_days"].median(),
    100 * (1 - subjects["event_flag"].mean())))

train, val, holdout = tte.split_subjects(subjects)
verify_no_lookup(train, val, holdout)
for name, s in (("train", train), ("val", val), ("holdout", holdout)):
    ev_rate = s["event_flag"].mean() * 100
    med = s["obs_days"].median()
    print(f"  {name}: {len(s):,} строк | событий {ev_rate:.2f}% | медиана obs {med:.1f} дн")
X_cols = tte.subject_features(train)
print(f"фич: {len(X_cols)}")


субъектов: 1,603,057 | колонок: 85
событие в 30д: 73.8% | медиана obs: 6.0 дн | цензур: 26.2%


[leak-check] ok: train/val <= 2025-06-30, holdout >= 2025-07-01, метки не пересекают границу
  train: 1,238,098 строк | событий 73.04% | медиана obs 6.2 дн
  val: 93,497 строк | событий 76.65% | медиана obs 5.5 дн
  holdout: 220,926 строк | событий 77.40% | медиана obs 5.5 дн
фич: 64


In [3]:
# 2. Сбалансированные выборки + person-time развёртка
tr_sub = tte.sample_subjects(train, CFG["n_starts"], CFG["n_fault"],
                             CFG["n_nonfault"], seed=CFG["seed"])
va_sub = tte.sample_subjects(val, CFG["n_val"] // 3, CFG["n_val"] // 3,
                             CFG["n_val"] // 3, seed=CFG["seed"] + 1)
ho_sub = tte.sample_holdout(holdout, CFG["n_hold"], seed=777)
print(f"train/val/holdout выборки: {len(tr_sub):,} / {len(va_sub):,} / {len(ho_sub):,}")
X_tr, y_tr = tte.expand_person_time(tr_sub, X_cols, tte.HORIZON_BUCKETS)
X_va, y_va = tte.expand_person_time(va_sub, X_cols, tte.HORIZON_BUCKETS)
print(f"person-time: train={X_tr.shape} val={X_va.shape} | событий в шаге: {y_tr.mean()*100:.3f}%")


train/val/holdout выборки: 35,000 / 6,000 / 19,999


person-time: train=(1285043, 69) val=(200610, 69) | событий в шаге: 2.335%


In [4]:
# 3. CatBoost discrete hazard: дефолты 1000/0.03/10 (заменяют 500/0.05/6)
t0 = time.time()
m = fit_discrete_hazard(X_tr, y_tr, X_va, y_va, iters=CFG["iters"],
                        lr=CFG["lr"], depth=CFG["depth"], seed=CFG["seed"],
                        task_type=CFG["task_type"])
print(f"[train] готово за {time.time()-t0:.0f}s, best_iteration={m.get_best_iteration()}")
m.save_model(MODELS / "tte_wear_discrete_hazard.cbm")

imp = pd.Series(m.get_feature_importance(type="PredictionValuesChange"),
                index=X_cols + ["шаг", "час_шага", "день_недели_шага",
                                "месяц_шага", "доля_горизонта"])
"\nТоп-15 фич (hazard):"
print(imp.sort_values(ascending=False).head(15).round(3).to_string())


0:	learn: 0.6182181	test: 0.6188574	best: 0.6188574 (0)	total: 57.3ms	remaining: 57.2s


100:	learn: 0.0717639	test: 0.0802207	best: 0.0802207 (100)	total: 3.06s	remaining: 27.3s


200:	learn: 0.0683611	test: 0.0790129	best: 0.0790124 (199)	total: 6.23s	remaining: 24.8s


300:	learn: 0.0665033	test: 0.0787173	best: 0.0787123 (298)	total: 9.15s	remaining: 21.3s


400:	learn: 0.0650446	test: 0.0786293	best: 0.0786289 (389)	total: 12.1s	remaining: 18.1s


...

700:	learn: 0.0616744	test: 0.0785306	best: 0.0785013 (658)	total: 20.9s	remaining: 8.89s


800:	learn: 0.0606178	test: 0.0785387	best: 0.0785013 (658)	total: 23.7s	remaining: 5.9s


900:	learn: 0.0596288	test: 0.0785787	best: 0.0785013 (658)	total: 27s	remaining: 2.97s


bestTest = 0.07850129858
bestIteration = 658
Shrink model to first 659 iterations.
[train] готово за 31s, best_iteration=658


час_шага                         15.038
шаг                              12.468
день_недели_шага                 10.483
доля_горизонта                   10.192
дни_с_посл_неиспр                 9.685
дни_с_посл_старта                 6.741
неисправностей_сум_30д            3.798
ид_объект_code                    2.344
тип_датчика_code                  2.068
неисправностей_сум_7д             1.766
событий                           1.751
z_событий                         1.630
день_года                         1.555
неисправностей_об_б_сум_30д       1.224
каналов_активных_об_б_сум_30д     1.172


In [5]:
# 4. Метрики на holdout >= 2025-07-01 (3-30 дней) + сохранение отчёта/предсказаний
surv_tr = tte.make_surv_struct(tr_sub["event_flag"], tr_sub["obs_days"])
surv_ho = tte.make_surv_struct(ho_sub["event_flag"], ho_sub["obs_days"])
S_ho = predict_survival_discrete(m, ho_sub, X_cols)
risk = 1.0 - S_ho[:, -1]
met = survival_metrics(surv_tr, surv_ho, risk, S_ho, tte.EVAL_TIMES)

step24 = int(round(1.0 * tte.STEPS_PER_DAY)) - 1
p24 = 1.0 - S_ho[:, step24]
y24 = ((ho_sub["event_flag"] == 1) & (ho_sub["obs_days"] <= 1.01)).to_numpy(int)
pr24 = pr_at_horizon(y24, p24)

# горизонты 48ч / 7д / 30д — как для заявок ТО
rows = []
for h, label in ((1.0, "24ч"), (2.0, "48ч"), (7.0, "7д"), (30.0, "30д")):
    yh = ((ho_sub["event_flag"] == 1) & (ho_sub["obs_days"] <= h + 0.01)).to_numpy(int)
    ph = 1.0 - S_ho[:, int(round(h * tte.STEPS_PER_DAY)) - 1]
    prh = pr_at_horizon(yh, ph)
    rows.append({"горизонт": label, "база": "%.2f%%" % (prh["base"] * 100),
                 "PR-AUC": round(prh["pr_auc"], 3)})
print("PR-AUC по горизонтам (holdout):")
print(pd.DataFrame(rows).to_string(index=False))

S_km, _ = baseline_survival_by_type(surv_tr, tr_sub["тип_датчика"].to_numpy(),
                                     ho_sub["тип_датчика"].to_numpy(), tte.EVAL_TIMES)
km = survival_metrics(surv_tr, surv_ho, 1 - S_km[:, -1], S_km, tte.EVAL_TIMES)
print("\nUno-C=%.3f | meanAUC=%.3f | IBS=%.3f | KM-якорь: Uno-C=%.3f IBS=%.3f" % (
    met["uno_c"], met["mean_auc"], met["ibs"], km["uno_c"], km["ibs"]))
print("P(24ч): PR-AUC=%.4f prec@r>=.5=%.4f база=%.2f%%" % (
    pr24["pr_auc"], pr24["prec@r>=.5"], pr24["base"] * 100))

out = ho_sub[["ид_канала_данных", "бакет", "дата", "тип_датчика",
              "event_flag", "obs_days"]].copy()
out["p24"] = p24
out["risk30"] = risk
out["exp_days"] = S_ho.sum(axis=1) / tte.STEPS_PER_DAY
out.to_parquet(MODELS / "tte_wear_holdout.parquet", index=False)

report = ("=========== TTE WEAR (defaults 1000/0.03/10, %s) ===========\n"
          "holdout >= 2025-07-01: %d субъектов\n"
          "Uno-C=%.3f | meanAUC=%.3f | IBS=%.3f | KM-якорь: Uno-C=%.3f IBS=%.3f\n"
          "P(24ч): PR-AUC=%.4f prec@r>=.5=%.4f база=%.1f%%\n"
          "PR-AUC по горизонтам: %s\n"
          % (CFG["task_type"], len(ho_sub), met["uno_c"], met["mean_auc"],
             met["ibs"], km["uno_c"], km["ibs"], pr24["pr_auc"],
             pr24["prec@r>=.5"], pr24["base"] * 100, rows))
(MODELS / "tte_wear_report.txt").write_text(report, encoding="utf-8")
print("\n[save] model/report/holdout -> models/tte_wear_*")


PR-AUC по горизонтам (holdout):
горизонт   база  PR-AUC
     24ч 12.73%   0.890
     48ч 20.82%   0.741
      7д 58.67%   0.930
     30д 76.96%   0.973



Uno-C=0.761 | meanAUC=0.839 | IBS=0.108 | KM-якорь: Uno-C=0.655 IBS=0.167
P(24ч): PR-AUC=0.8900 prec@r>=.5=0.9945 база=12.73%

[save] model/report/holdout -> models/tte_wear_*


In [6]:
# 5. Inference-контракт (P0): единый API поверх моделей
#   predict_risk -> p24/risk30/exp_days; топ-K по risk30 (калибр. там, где есть)
print("калибраторы:", {t: ic.is_calibrated(t) for t in ic.TASKS})

# Wear: пересчёт вероятностей через контракт на подвыборке holdout
pr = ic.predict_risk(m, ho_sub.iloc[:1000])
print("\nwear predict_risk (n=1000):")
print(pr.describe().round(4).to_string())

# топ-K по p24 (24ч-прецизия) для wear
y24w = ((ho_sub["event_flag"] == 1) & (ho_sub["obs_days"] <= 1.01)).to_numpy(int)
tmp = ho_sub.assign(p24=out["p24"])
tbl = pd.DataFrame([ic.top_k(tmp, "p24", k, y24w)
                    for k in (50, 100, 500, 2000)])
print("\nтоп-K wear (p24, 24ч):")
print(tbl.round(4).to_string(index=False))


калибраторы: {'fire': False, 'access': True, 'sensor': True, 'wear': False}

wear predict_risk (n=1000):
             p24     risk30   exp_days
count  1000.0000  1000.0000  1000.0000
mean      0.1243     0.7741    11.3790
std       0.2900     0.3236     9.3152
min       0.0001     0.0087     0.0066
25%       0.0014     0.6746     4.2583
50%       0.0079     0.9605     8.1067
75%       0.0583     0.9999    17.5561
max       0.9987     1.0000    29.8504

топ-K wear (p24, 24ч):
   K  precision  recall  hits  total
  50     1.0000  0.0196    50   2545
 100     1.0000  0.0393   100   2545
 500     0.9940  0.1953   497   2545
2000     0.9565  0.7517  1913   2545


In [7]:
# 6. Финальная таблица метрик (P0) — по сохранённым holdout-предсказаниям
#    (sensor пока на старой панели; обновится в notebook 27 после пересборки)
ft = ic.final_metrics()
ft.to_csv(DATASET / "final_metrics_v0.csv", index=False, encoding="utf-8-sig")
print(ft.round(4).to_string(index=False))
print("\nсохранено: dataset/final_metrics_v0.csv")


  task  n_holdout  base24  pr_auc_24  prec@50  prec@100  recall@2000  K@prec>=.7(24h)  ece_risk30_raw  ece_risk30_cal  median_exp_days  uno_c   ibs  km_uno_c
  fire      19999  0.0153     0.4206     1.00      0.97       0.7124            152.0          0.0781             NaN          28.4144  0.792 0.055     0.723
access      19999  0.0101     0.0304     0.00      0.01       0.3317              NaN          0.0212          0.0152          28.3990  0.794 0.054     0.658
sensor      19999  0.0031     0.0159     0.02      0.01       0.2951              NaN          0.0251          0.0208          29.1195  0.710 0.014     0.643
  wear      19999  0.1273     0.8900     1.00      1.00       0.7517           3035.0          0.0195             NaN           8.6242  0.761 0.108     0.655

сохранено: dataset/final_metrics_v0.csv


In [8]:
# 7. Drift-мониторинг (P1): процентили p24/risk30 по кварталам holdout
for task in ("fire", "access", "sensor", "wear"):
    try:
        d = ic.risk_drift_by_quarter(task)
        print(f"--- {task} ---")
        print(d.round(4).to_string(index=False))
        print()
    except FileNotFoundError:
        print(f"--- {task}: нет артефактов, пропуск ---")


--- fire ---
квартал  p24_mean  p24_median  p24_p95  risk30_mean  risk30_median  risk30_p95  risk30_cal_mean  risk30_cal_median  risk30_cal_p95
 2025Q3    0.0149      0.0009   0.0602       0.1134         0.0094      0.7865           0.1134             0.0094          0.7865
 2025Q4    0.0107      0.0041   0.0381       0.1653         0.0996      0.5923           0.1653             0.0996          0.5923
 2026Q1    0.0172      0.0065   0.0554       0.2562         0.2710      0.6778           0.2562             0.2710          0.6778
 2026Q2    0.0143      0.0031   0.0615       0.1714         0.0525      0.6248           0.1714             0.0525          0.6248

--- access ---
квартал  p24_mean  p24_median  p24_p95  risk30_mean  risk30_median  risk30_p95  risk30_cal_mean  risk30_cal_median  risk30_cal_p95
 2025Q3    0.0053      0.0023   0.0192       0.1947         0.0915      0.8841           0.1925             0.0933          0.8802
 2025Q4    0.0059      0.0026   0.0213       0.2270   

In [9]:
# 8. P1-быстрое: time-of-day / weekend профиль охранных тревог (access)
#    — есть ли выраженный "сонный час" / выходные, на которые стоит опираться?
ho_acc = pd.read_parquet(MODELS / "tte_access_holdout.parquet")
b = pd.to_datetime(ho_acc["бакет"] * tte.hour_ns())
hoa = ho_acc.assign(час_бакета=b.dt.hour, день_недели=b.dt.dayofweek)
ev = (hoa["event_flag"] == 1) & (hoa["obs_days"] <= 7.01)
tbl = hoa[ev].groupby("час_бакета")["ид_канала_данных"].count()
print("Распределение охранных событий (<=7д, holdout) по часу бакета:")
print(tbl.to_string())
week = hoa[ev].groupby("день_недели")["ид_канала_данных"].count()
print("\nпо дню недели (0=пн):")
print(week.to_string())
print("\nВывод для фич: база доступа (час/день недели) — кандидат в колонки панели;"
      " прод-инференс на это значение пока не опирается.")


Распределение охранных событий (<=7д, holdout) по часу бакета:
час_бакета
0      68
6     921
12    698
18    208

по дню недели (0=пн):
день_недели
0    319
1    354
2    356
3    360
4    297
5    126
6     83

Вывод для фич: база доступа (час/день недели) — кандидат в колонки панели; прод-инференс на это значение пока не опирается.
